# 🎮 Reverse Survivor: The Director — Обучение ИИ в Google Colab

### ⚠️ ПЕРЕД СТАРТОМ: Проверьте настройки среды:
1. В верхнем меню Colab выберите: **Среда выполнения (Runtime) -> Сменить среду выполнения (Change runtime type)**.
2. **Тип среды выполнения**: выберите **`Python 3`** *(НЕ выбирайте `R`!)*.
3. **Аппаратный ускоритель**: выберите **`Графический процессор T4 (T4 GPU)`** *(НЕ выбирайте `TPU`!)*.
4. Нажмите **Сохранить**.

In [ ]:
# [Шаг 1] Проверка видеокарты NVIDIA T4
!nvidia-smi

In [ ]:
# [Шаг 2] Установка компилятора Rust и Cargo
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] += ":/root/.cargo/bin"
!rustc --version

In [ ]:
# [Шаг 3] Загрузка и распаковка архива игры в Colab
from google.colab import files
import zipfile, os, shutil

# Очищаем и создаем стандартную папку Colab: /content/Anti_VS
if os.path.exists('/content/Anti_VS'):
    shutil.rmtree('/content/Anti_VS')
os.makedirs('/content/Anti_VS', exist_ok=True)

print("Нажмите кнопку 'Choose Files' ниже и выберите файл Anti_VS_Colab.zip (из вашей папки C:\\Anti_VS):")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            for member in zip_ref.infolist():
                clean_path = member.filename.replace('\\', '/').lstrip('/')
                target_file = os.path.join('/content/Anti_VS', clean_path)
                if member.is_dir() or clean_path.endswith('/'):
                    os.makedirs(target_file, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target_file), exist_ok=True)
                    with open(target_file, 'wb') as f:
                        f.write(zip_ref.read(member.filename))
        print("\n✅ Проект успешно распакован в /content/Anti_VS!")
        break

# Переходим в папку проекта через Colab magic %cd
%cd /content/Anti_VS

# Проверяем структуру файлов
!ls -la /content/Anti_VS
!ls -la /content/Anti_VS/src

In [ ]:
# [Шаг 4] Проверка сборки и запуск турнира Со-Эволюции (League Self-Play)
%cd /content/Anti_VS

# Проверка зависимостей
!cargo check --release

# Запуск турнира (например, 20 поколений, 12 кандидатов)
!cargo run --release -- --train 20 12

In [ ]:
# [Шаг 5] Скачивание обученных моделей обратно на ваш компьютер
import os
from google.colab import files

# Используем абсолютные пути файловой системы Colab
boss_path = '/content/Anti_VS/models/boss_policy.json'
director_path = '/content/Anti_VS/models/director_policy.json'

if os.path.exists(boss_path):
    files.download(boss_path)
    print("✅ Модель Босса скачана! Перенесите её в вашу папку: C:\\Anti_VS\\models\\boss_policy.json")
else:
    print(f"❌ Файл {boss_path} не найден! Убедитесь, что Шаг 4 завершился успешно.")

if os.path.exists(director_path):
    files.download(director_path)
    print("✅ Модель Директора скачана! Перенесите её в: C:\\Anti_VS\\models\\director_policy.json")
